In [1]:
%run stdPackages.ipynb
os.chdir(d['py'])
from adjustIO import SimpleBalance
from modelData import *

No clean-up of work-folder


# Make data for PCd model

## 1. Load

Load + main settings:

In [2]:
name = 'PCdDb'
ioName = 'IO_OPTdb' # name of database
f_IOdata = os.path.join(d['project'], 'IOdata','data','processedData', ioName) # points to relevant raw database
db = GpyDB(f_IOdata, name = name, ws = d['work'])
db0 = db.copy() # store copy of original data
ws = db.ws # use this as the main workspace throughout
t0 = 2019
T  = t0+5

## 2. Clean data

The current implementation does not include several sectors. Here, we start by simply removing them and then re-balance industries' demand to balance this out.

### A. Clean removals

Remove industries:

In [3]:
delS = reduce(pd.Index.union, (db('s_f'), db('s_G'), pd.Index(['itory'],name='s')))
AggDB.subsetDB(db, db('s').difference(delS))

Remove goods:

In [4]:
delN = reduce(pd.Index.union, [db('n_F')])
AggDB.subsetDB(db, db('n').difference(delN))

Set taxes to zero in the baseline:

In [5]:
db['TotalTax'] = pd.Series(0, index = db('TotalTax').index)

Clean other symbols:

In [6]:
[db.series.__delitem__(k) for k in ('n_F','s_G','s_f','qCO2','M1990','vTax','taxTypes',
                                    'vD_dur','vD_depr','vD_inv') if k in db.symbols];

### B. Balance data

We use the ```SimpleBalance``` to balance the IO with the following inputs:
* ```vD0, vS0```: Initial data vectors (demand and supply).
* ```vDBar, vSBar```: Fixed elements in the demand/supply vectors. We fix the element ```resIncome``` = 0 for all industries to target zero profits.
* ```vDT, vST```: Target vectors in balancing (use default).
* ```nEQ```: What markets to ensure equilibrium on (use default).
* ```sT```: In what sectors do we target a specific balance. With ```resIncome``` fixed at some level, this is used to target profit margins. If we want to take taxes into consideration, this has to be added as a component in the "demand" vector. Here, we target zero profits in the two production industries, not for the household sector (they may have other income, e.g. return on assets).

*Note: We can add things like bounds on solution variables or load factors in the objective by adjusting relevant parameters.*

Fix residual income at long run return on durables and investments at steady state levels:

In [7]:
profitsIdx = pd.Index(['resIncome'], name = 'n')
profits = pd.Series(0, index = adj.rc_pd(db('vD'), profitsIdx).index) + (db('pD_dur') * adj.rc_pd(db('qD'), db('dur_p'))).groupby('s').sum()
invests = adjMultiIndex.applyMult(adj.rc_pd(db('vD'), db('dur_p')) * (db('rDepr')+db('g_LR')), db('dur2inv')).droplevel('n').rename_axis(['s','n'])
vD_noK = adj.rc_pd(db('vD'), ('not', db('dur_p')))
vD_K = adj.rc_pd(db('vD'), db('dur_p'))

Balance data:

In [8]:
b = SimpleBalance(ws = ws)(vD_noK, db('vS'), 
                           vDBar = profits.combine_first(invests),
                           sT = db('s_p').union(db('s_i')))

*Update data:*

In [9]:
db['vD'] = adj.rc_pd(b('vDsolve'), ('not', profitsIdx)).rename('vD')
db['vS'] = b('vSsolve').rename('vS')
db['qD'] = db('vD') / db('p')
db['qS'] = db('vS') / db('p')
# Add durables again
db.aom(vD_K, name = 'vD')
db.aom(vD_K, name = 'qD')

## 3. Production Module

### A. Nesting

Add a bit of nesting structure - split out industry inputs in one branch and labor in a second:

In [11]:
m = reduce(pd.Index.union, [pd.MultiIndex.from_tuples([('Y','IND'), ('Y','KL'), ('KL','L'), ('KL','K')], names = ['n','nn']),
                            pd.MultiIndex.from_product([['IND'], db('n_p')+'_input'], names = ['n','nn'])]
          )
m = pyDatabases.cartesianProductIndex([db('s_p'), m]) # repeat for all industries
# sparsity:
sparsity = adj.rc_pd(db('vD')[db('vD')!=0], db('s_p')).index # inputs that different industries demand
mapNames = pd.Series(db('n_p')+'_input', index = db('n_p')).combine_first(pd.Series(sparsity.levels[-1], index = sparsity.levels[-1]))
sparsity = sparsity.set_levels(sparsity.levels[-1].map(mapNames), level = -1)
m = nestingTree.trimNestingStructure(m, sparsity) # trim nesting structure with demand

Replace ```Y``` with industry name:

In [13]:
df = m.to_frame(index=False)
df.loc[df.n == 'Y','n'] = df.loc[df.n == 'Y', 's']
m = pd.MultiIndex.from_frame(df)
db['nestProduction'] = m

### B. Elasticities

Add elasticities of substitution in the two nests:

In [15]:
σY = pd.Series(0.6, index = m[m.get_level_values('s') == m.get_level_values('n')].droplevel('nn').unique(), name = 'sigma')
σKL = pd.Series(0.5, index = m[m.get_level_values('n') == 'KL'].droplevel('nn').unique(), name = 'sigma')
σIND = pd.Series(0.4, index = m[m.get_level_values('n') == 'IND'].droplevel('nn').unique(), name = 'sigma')
db.aom(pd.concat([σY, σIND,σKL], axis = 0), name = 'sigma')

### C. Regulation

Specify tax rates and define effective marginal costs/prices:

In [17]:
db.aom(pd.Series(0, index = adj.rc_pd(db('qD'), db('s_p')).index, name = 'tauD'))
db.aom(pd.Series(0, index = adj.rc_pd(db('qS'), db('s_p')).index, name = 'tauS'))
db.aom(adj.rc_pd(db('TotalTax'), db('s_p')), name = 'tauLump')
db.aom(adj.rc_pd(((1+db('tauD'))*db('p')).dropna(), db('s_p')), name = 'pD')
db.aom(adj.rc_pd(((1-db('tauS'))*db('p')).dropna(), db('s_p')), name = 'pS')

## 4. Households

### A. Nesting

In [18]:
m = pyDatabases.cartesianProductIndex([db('s_HH'), pd.MultiIndex.from_product([['C'], db('n_p')], names = ['n','nn'])])
sparsity = adj.rc_pd(db('vD')[db('vD')!=0], db('s_HH')).index
m = nestingTree.trimNestingStructure(m, sparsity)
db['nestHH'] = m

Add mapping from consumption aggregate to labor:

In [19]:
db['L2C'] = pd.MultiIndex.from_arrays([db('s_HH'), pd.Index(['L']*len(db('s_HH')), name = 'n'), pd.Index(['C']*len(db('s_HH')))], names = ['s','n','nn'])

### B. Preferences

In [20]:
db.aom(pd.Series(0.5, index = m[m.get_level_values('n') == 'C'].droplevel('nn').unique(), name = 'sigma'), name = 'sigma') # EOS, intratemporal
db.aom(pd.Series(0.1, index = db('s_HH')), name = 'frisch') # frisch elasticity, if endogenous labor supply is turned on
db.aom(pd.Series(2, index = db('s_HH')), name = 'crra') # 1/IES
db.aom(pd.Series((1+db('g_LR'))**(db('crra'))/db('R_LR'), index = db('s_HH')), name = 'discF') # time preference on balanced growth path

### C. Regulation:

No marginal taxes:

In [21]:
db.aom(pd.Series(0, index = adj.rc_pd(db('qD'), db('s_HH')).index, name = 'tauD'))
db.aom(pd.Series(0, index = adj.rc_pd(db('qS'), db('s_HH')).index, name = 'tauS'))
db.aom(adj.rc_pd(db('TotalTax'), db('s_HH')), name = 'tauLump')
db.aom(adj.rc_pd(((1+db('tauD'))*db('p')).dropna(), db('s_HH')), name = 'pD')
db.aom(adj.rc_pd(((1-db('tauS'))*db('p')).dropna(), db('s_HH')), name = 'pS')

Initialize assets (this one without owner shares in domestic firms)

In [22]:
db.aom(pd.Series(0, index = db('s_HH')), name = 'vA')

## 5. Investment module TO DO

## X. Other

Dummy for industries that enter government budget:

In [23]:
db['d_TotalTax'] = db['TotalTax'].index

Add time to most:

In [24]:
def add_t0(k):
    k.index = stdSort(pd.MultiIndex.from_frame(k.index.to_frame(index=False).assign(t = t0))) 
[add_t0(db(k)) for k in db.getTypes(['var','par']) if k not in ('sigma','frisch','crra','discF')]; # for all variables and parameters in the database, add the 't' dimension:
addTimeToDB(t0, T, db) # add a number of relevant time subsets 

Other parameters/variables used throughout:

In [25]:
db.aom(pd.Series(db('R_LR'), index = db('t')), name = 'Rrate', priority = 'first') # fix interest rate path at long run level
db.aom(pd.Series(db('R_LR')*(1+db('infl_LR')), index = db('t')), name = 'iRate', priority = 'first')

## Export

In [26]:
db.export(repo=d['data'])